# 01 — Is the CLIP text encoder MDM conditions on blind to spatial language?

**Question.** MDM's text conditioning is a frozen CLIP ViT-B/32 text encoder (`model/mdm.py`,
`load_and_freeze_clip`). Does that encoder actually distinguish spatial/directional language
("left" vs "right", "forward" vs "backward") at all, or does it embed such minimal pairs as
near-identical — which would mean the *conditioning signal itself* cannot tell the diffusion
model which direction was asked for, independent of anything about the generator or the dataset?

**Why this matters here.** Per `docs/DECISIONS.md` D-29: before running any more training
comparisons (which repeat E1's exact shape — an unproven training budget, scored by an instrument
already under suspicion), the instruments themselves need checking. This notebook checks one
candidate root cause for the diffusion model's own weak-conditioning behavior observed all
session (`docs/EXPERIMENT_LOG.md` E1B) — not by training anything, by directly probing the frozen
encoder used in the real pipeline.

**Design history (two rounds of review, both incorporated, not glossed over):**
- **Round 1 (SUP-20260907-82):** the original two-group design (spatial *modifier* pairs vs
  non-spatial *verb* pairs) confounded "CLIP is blind to spatial language" with the weaker
  "CLIP separates verbs better than modifiers in general." Fixed by adding a third group:
  non-spatial *modifier* pairs in the same syntactic slot as the spatial group. Result: the
  spatial-vs-modifier comparison isolated the effect (verb-vs-modifier showed no difference,
  p=0.097 — the load-bearing check that the effect is about spatial-ness, not word class).
- **Round 2 (SUP-20260907-83):** that isolated result was real but **underpowered at n=16/group**
  (rank-biserial 0.383, Cohen's d ~0.68-0.83 depending on the conversion used — see the power
  section below), and the reported significance depended on switching a t-test from two-sided
  (p=0.070, not significant) to one-sided (p=0.035) *after* seeing the two-sided result — the
  correct call given a genuinely pre-registered direction, but reported in a way that didn't show
  the two-sided number alongside it, which reads as the shape of p-hacking regardless of intent.
  **This round fixes both:** every group expanded from 16 to 40 pairs (CLIP text embedding is
  instant — no compute, no training, no dataset needed to fix an underpowered test here, unlike
  E1's affordability problem), every comparison below reports **both** one-sided and two-sided p,
  and a Bonferroni threshold for the 3 pairwise comparisons actually run (0.05/3 = 0.0167) is
  stated explicitly rather than left implicit.

**Instrument.** MDM's own CLIP loader and tokenization convention, copied verbatim from
`model/mdm.py::MDM.load_and_freeze_clip` / `MDM.clip_encode_text` — not reimplemented, not a
stand-in. `ViT-B/32`, the version `utils/model_util.py` hardcodes for every checkpoint in this
project.

**Literature context** (`guidance/VERIFICATION_NOTE.md`, independently re-verified before this
notebook was written): arXiv:2311.11477, "What's left can't be right — the remaining positional
incompetence of contrastive vision-language models," confirms the *direction* of this finding for
CLIP-style models generally. The "BLIP scored 56% vs 99% human" statistic sometimes attached to
that paper belongs to a different paper (**not cited here**).

In [1]:
import os
import sys

MDM_ROOT = os.path.join("..", "third_party", "motion-diffusion-model")
sys.path.insert(0, os.path.abspath(MDM_ROOT))

import numpy as np
import torch
import clip
from scipy import stats

torch.manual_seed(0)
print("torch:", torch.__version__)
print("clip available models:", clip.available_models())

torch: 2.13.0
clip available models: ['RN50', 'RN101', 'RN50x4', 'RN50x16', 'RN50x64', 'ViT-B/32', 'ViT-B/16', 'ViT-L/14', 'ViT-L/14@336px']


## 1. Load MDM's own CLIP text encoder, exactly as `model/mdm.py` does

In [2]:
def load_and_freeze_clip(clip_version):
    # Copied verbatim from third_party/motion-diffusion-model/model/mdm.py::MDM.load_and_freeze_clip
    clip_model, clip_preprocess = clip.load(clip_version, device='cpu', jit=False)
    clip.model.convert_weights(clip_model)
    clip_model.eval()
    for p in clip_model.parameters():
        p.requires_grad = False
    return clip_model


def clip_encode_text(clip_model, raw_text, dataset="humanml"):
    # Copied verbatim from third_party/motion-diffusion-model/model/mdm.py::MDM.clip_encode_text
    device = "cpu"
    max_text_len = 20 if dataset in ["humanml", "kit"] else None
    if max_text_len is not None:
        default_context_length = 77
        context_length = max_text_len + 2
        assert context_length < default_context_length
        texts = clip.tokenize(raw_text, context_length=context_length, truncate=True).to(device)
        zero_pad = torch.zeros([texts.shape[0], default_context_length - context_length],
                                dtype=texts.dtype, device=texts.device)
        texts = torch.cat([texts, zero_pad], dim=1)
    else:
        texts = clip.tokenize(raw_text, truncate=True).to(device)
    with torch.no_grad():
        return clip_model.encode_text(texts).float()


clip_model = load_and_freeze_clip("ViT-B/32")
print("CLIP text encoder loaded and frozen.")

CLIP text encoder loaded and frozen.


## 2. Minimal pairs — three matched groups, n=40 each

**Pilot set (n=16, unchanged from round 1)** kept first and separately, so the originally-audited
pairs stay auditable. **Expansion set (n=24 new pairs per group)** appended after, varying
sentence *frames* (not just re-using the same template with different words), per
SUP-20260907-83's requirement that added n buy independent draws, not correlated ones.

- **Spatial:** substituted word is a directional/positional modifier.
- **Non-spatial verb:** substituted word is a verb.
- **Non-spatial modifier:** substituted word is a modifier (adjective/adverb), same syntactic
  slot as the spatial group, never directional.

In [3]:
SPATIAL_PAIRS_PILOT = [
    ("a person raises their left arm", "a person raises their right arm"),
    ("a person turns left", "a person turns right"),
    ("a person walks forward", "a person walks backward"),
    ("a person steps over the box", "a person steps around the box"),
    ("a person kicks with their left leg", "a person kicks with their right leg"),
    ("a person moves forward", "a person moves backward"),
    ("a person walks to the left", "a person walks to the right"),
    ("a person raises their left hand", "a person raises their right hand"),
    ("a person leans forward", "a person leans backward"),
    ("a person jumps forward", "a person jumps backward"),
    ("a person waves with their left hand", "a person waves with their right hand"),
    ("a person spins clockwise", "a person spins counterclockwise"),
    ("a person steps forward", "a person steps backward"),
    ("a person reaches with their left arm", "a person reaches with their right arm"),
    ("a person walks in front of the chair", "a person walks behind the chair"),
    ("a person bends their left knee", "a person bends their right knee"),
]

SPATIAL_PAIRS_EXPANSION = [
    ("the person points to the left", "the person points to the right"),
    ("a person swings their left arm", "a person swings their right arm"),
    ("a person tilts their head left", "a person tilts their head right"),
    ("a person crouches and moves forward", "a person crouches and moves backward"),
    ("a person hops to the left side", "a person hops to the right side"),
    ("a person extends their left leg", "a person extends their right leg"),
    ("a person circles clockwise around the room", "a person circles counterclockwise around the room"),
    ("a person shifts their weight to the left", "a person shifts their weight to the right"),
    ("a person steps in front of the table", "a person steps behind the table"),
    ("a person drags their left foot", "a person drags their right foot"),
    ("a person twists to the left", "a person twists to the right"),
    ("a person marches forward in a line", "a person marches backward in a line"),
    ("a person lifts their left shoulder", "a person lifts their right shoulder"),
    ("a person walks backward slowly", "a person walks forward slowly"),
    ("a person rotates their body clockwise", "a person rotates their body counterclockwise"),
    ("a person stretches their left arm out", "a person stretches their right arm out"),
    ("a person moves to the left quickly", "a person moves to the right quickly"),
    ("a person kicks forward with force", "a person kicks backward with force"),
    ("a person glances to the left", "a person glances to the right"),
    ("a person stands behind the door", "a person stands in front of the door"),
    ("a person hops forward twice", "a person hops backward twice"),
    ("a person pivots on their left foot", "a person pivots on their right foot"),
    ("a person raises their left knee high", "a person raises their right knee high"),
    ("a person leans their body left", "a person leans their body right"),
]

NONSPATIAL_VERB_PAIRS_PILOT = [
    ("a person walks forward", "a person runs forward"),
    ("a person sits on a chair", "a person stands on a chair"),
    ("a person waves with their hand", "a person claps with their hand"),
    ("a person kicks the ball", "a person throws the ball"),
    ("a person picks up the box", "a person drops the box"),
    ("a person walks quickly", "a person walks slowly"),
    ("a person jumps once", "a person jumps twice"),
    ("a person sits on the chair", "a person sits on the table"),
    ("a person claps their hands", "a person waves their hands"),
    ("a person walks forward", "a person skips forward"),
    ("a person reads a book", "a person writes a letter"),
    ("a person opens the door", "a person closes the door"),
    ("a person drinks water", "a person eats food"),
    ("a person plays the guitar", "a person plays the piano"),
    ("a person laughs loudly", "a person cries softly"),
    ("a person dances happily", "a person dances sadly"),
]

NONSPATIAL_VERB_PAIRS_EXPANSION = [
    ("a person bakes a cake", "a person cooks a meal"),
    ("a person sweeps the floor", "a person mops the floor"),
    ("a person paints the wall", "a person cleans the wall"),
    ("a person ties their shoes", "a person removes their shoes"),
    ("a person folds the clothes", "a person washes the clothes"),
    ("a person waters the plant", "a person trims the plant"),
    ("a person types on the keyboard", "a person clicks the mouse"),
    ("a person hums a tune", "a person whistles a tune"),
    ("a person nods their head", "a person shakes their head"),
    ("a person yawns quietly", "a person sneezes quietly"),
    ("a person stretches their body", "a person relaxes their body"),
    ("a person balances on one foot", "a person hops on one foot"),
    ("a person catches the frisbee", "a person throws the frisbee"),
    ("a person unwraps the gift", "a person wraps the gift"),
    ("a person waves goodbye", "a person bows goodbye"),
    ("a person sharpens the pencil", "a person breaks the pencil"),
    ("a person feeds the dog", "a person walks the dog"),
    ("a person answers the phone", "a person ignores the phone"),
    ("a person locks the door", "a person unlocks the door"),
    ("a person tastes the soup", "a person stirs the soup"),
    ("a person signs the paper", "a person reads the paper"),
    ("a person lifts the weight", "a person drops the weight"),
    ("a person hugs their friend", "a person greets their friend"),
    ("a person whispers a secret", "a person shouts a secret"),
]

NONSPATIAL_MODIFIER_PAIRS_PILOT = [
    ("a person raises their arm slowly", "a person raises their arm quickly"),
    ("a person walks slowly", "a person walks quickly"),
    ("a person raises their broken arm", "a person raises their injured arm"),
    ("a person kicks the red ball", "a person kicks the blue ball"),
    ("a person wears a blue shirt", "a person wears a red shirt"),
    ("a person picks up the small box", "a person picks up the large box"),
    ("a person carries a heavy bag", "a person carries a light bag"),
    ("a person claps their hands loudly", "a person claps their hands softly"),
    ("a person sits on the wooden chair", "a person sits on the metal chair"),
    ("a person throws the heavy ball", "a person throws the light ball"),
    ("a person dances gracefully", "a person dances clumsily"),
    ("a person raises their tired arm", "a person raises their strong arm"),
    ("a person walks on the wet floor", "a person walks on the dry floor"),
    ("a person holds the empty cup", "a person holds the full cup"),
    ("a person wears an old hat", "a person wears a new hat"),
    ("a person speaks quietly", "a person speaks loudly"),
]

NONSPATIAL_MODIFIER_PAIRS_EXPANSION = [
    ("a person wears a striped shirt", "a person wears a plain shirt"),
    ("a person eats a spicy meal", "a person eats a bland meal"),
    ("a person sits in a comfortable chair", "a person sits in an uncomfortable chair"),
    ("a person carries a fragile vase", "a person carries a sturdy vase"),
    ("a person drives an expensive car", "a person drives a cheap car"),
    ("a person reads a thick book", "a person reads a thin book"),
    ("a person wears a tight jacket", "a person wears a loose jacket"),
    ("a person paints a colorful picture", "a person paints a dull picture"),
    ("a person tells a funny joke", "a person tells a serious joke"),
    ("a person walks with a confident stride", "a person walks with a nervous stride"),
    ("a person holds a warm cup", "a person holds a cold cup"),
    ("a person wears a shiny ring", "a person wears a dull ring"),
    ("a person plays a difficult song", "a person plays an easy song"),
    ("a person climbs a steep hill", "a person climbs a gentle hill"),
    ("a person tells a short story", "a person tells a long story"),
    ("a person buys a fresh apple", "a person buys a rotten apple"),
    ("a person wears a formal dress", "a person wears a casual dress"),
    ("a person builds a tall tower", "a person builds a short tower"),
    ("a person sings a cheerful song", "a person sings a gloomy song"),
    ("a person eats a sweet candy", "a person eats a sour candy"),
    ("a person wears a heavy coat", "a person wears a thin coat"),
    ("a person drinks a hot coffee", "a person drinks a cold coffee"),
    ("a person owns a modern house", "a person owns an old house"),
    ("a person gives a clear explanation", "a person gives a confusing explanation"),
]

SPATIAL_PAIRS = SPATIAL_PAIRS_PILOT + SPATIAL_PAIRS_EXPANSION
NONSPATIAL_VERB_PAIRS = NONSPATIAL_VERB_PAIRS_PILOT + NONSPATIAL_VERB_PAIRS_EXPANSION
NONSPATIAL_MODIFIER_PAIRS = NONSPATIAL_MODIFIER_PAIRS_PILOT + NONSPATIAL_MODIFIER_PAIRS_EXPANSION

print(f"Spatial: {len(SPATIAL_PAIRS)} ({len(SPATIAL_PAIRS_PILOT)} pilot + {len(SPATIAL_PAIRS_EXPANSION)} expansion)")
print(f"Nonspatial-verb: {len(NONSPATIAL_VERB_PAIRS)} ({len(NONSPATIAL_VERB_PAIRS_PILOT)} pilot + {len(NONSPATIAL_VERB_PAIRS_EXPANSION)} expansion)")
print(f"Nonspatial-modifier: {len(NONSPATIAL_MODIFIER_PAIRS)} ({len(NONSPATIAL_MODIFIER_PAIRS_PILOT)} pilot + {len(NONSPATIAL_MODIFIER_PAIRS_EXPANSION)} expansion)")

Spatial: 40 (16 pilot + 24 expansion)
Nonspatial-verb: 40 (16 pilot + 24 expansion)
Nonspatial-modifier: 40 (16 pilot + 24 expansion)


In [4]:
def cosine_sim(a, b):
    a = a / a.norm(dim=-1, keepdim=True)
    b = b / b.norm(dim=-1, keepdim=True)
    return float((a * b).sum(dim=-1))


def pair_similarities(pairs):
    sims = []
    for s1, s2 in pairs:
        e1 = clip_encode_text(clip_model, [s1])
        e2 = clip_encode_text(clip_model, [s2])
        sims.append(cosine_sim(e1[0], e2[0]))
    return np.array(sims)


spatial_sims = pair_similarities(SPATIAL_PAIRS)
verb_sims = pair_similarities(NONSPATIAL_VERB_PAIRS)
modifier_sims = pair_similarities(NONSPATIAL_MODIFIER_PAIRS)

print(f"Computed {len(spatial_sims)} spatial, {len(verb_sims)} verb, {len(modifier_sims)} modifier similarities.")
print(f"Spatial mean so far: {spatial_sims.mean():.4f}, pilot-only mean: {spatial_sims[:16].mean():.4f}")

Computed 40 spatial, 40 verb, 40 modifier similarities.
Spatial mean so far: 0.9707, pilot-only mean: 0.9654


## 3. Power analysis, using the pilot's own effect size to justify the n=40 target

**Honesty about sequencing:** this is not a blind pre-registration written before any data
existed — the pilot (n=16/group, round 1) already showed an effect. What follows is the correct
thing to do *after* an underpowered pilot when more data costs nothing: use the pilot's own
effect size to compute what n is actually needed, then collect that much before drawing a
conclusion — not to keep adding pairs until something clears 0.05.

In [5]:
pilot_spatial = np.array(pair_similarities(SPATIAL_PAIRS_PILOT))
pilot_modifier = np.array(pair_similarities(NONSPATIAL_MODIFIER_PAIRS_PILOT))

n1, n2 = len(pilot_spatial), len(pilot_modifier)
pooled_sd = np.sqrt(((n1-1)*pilot_spatial.std(ddof=1)**2 + (n2-1)*pilot_modifier.std(ddof=1)**2) / (n1+n2-2))
pilot_d = (pilot_spatial.mean() - pilot_modifier.mean()) / pooled_sd
print(f"Pilot (n={n1}/group) direct Cohen's d (spatial vs modifier): {pilot_d:.3f}")

try:
    from statsmodels.stats.power import TTestIndPower
    power_calc = TTestIndPower()
    for power_target in [0.80, 0.95]:
        n_needed = power_calc.solve_power(effect_size=pilot_d, power=power_target, alpha=0.05, alternative='larger')
        print(f"  n/group needed for {int(power_target*100)}% power (one-sided alpha=0.05): {n_needed:.1f}")
    power_at_40 = power_calc.solve_power(effect_size=pilot_d, nobs1=40, alpha=0.05, alternative='larger')
    print(f"  achieved power at n=40/group: {power_at_40:.3f}")
except ImportError:
    print("statsmodels not installed -- reporting effect size only, no power curve.")

BONFERRONI_ALPHA = 0.05 / 3  # three pairwise comparisons run below
print(f"\nBonferroni-corrected alpha for 3 pairwise comparisons: {BONFERRONI_ALPHA:.4f}")

Pilot (n=16/group) direct Cohen's d (spatial vs modifier): 0.678


  n/group needed for 80% power (one-sided alpha=0.05): 27.6
  n/group needed for 95% power (one-sided alpha=0.05): 47.8
  achieved power at n=40/group: 0.913

Bonferroni-corrected alpha for 3 pairwise comparisons: 0.0167


## 4. Full-sample (n=40/group) comparisons — both sided p-values, Bonferroni threshold stated

Per SUP-20260907-83: report **both** one-sided and two-sided p for every comparison. The
one-sided alternative (`greater`) matches the pre-registered direction (spatial pairs expected to
be MORE similar / less separated than either control); the two-sided p is shown alongside it, not
substituted for it.

In [6]:
def describe(name, arr):
    print("%-20s (n=%2d): mean=%.4f  std=%.4f  min=%.4f  max=%.4f" % (
        name, len(arr), arr.mean(), arr.std(ddof=1), arr.min(), arr.max()))

describe("Spatial", spatial_sims)
describe("Nonspatial-verb", verb_sims)
describe("Nonspatial-modifier", modifier_sims)

h_stat, h_p = stats.kruskal(spatial_sims, verb_sims, modifier_sims)
print(f"\nKruskal-Wallis (all 3 groups, n=40 each): H={h_stat:.3f}, p={h_p:.6f}")

def compare(name_a, a, name_b, b):
    t_two = stats.ttest_ind(a, b, equal_var=False, alternative="two-sided")
    t_one = stats.ttest_ind(a, b, equal_var=False, alternative="greater")
    u_two = stats.mannwhitneyu(a, b, alternative="two-sided")
    u_one = stats.mannwhitneyu(a, b, alternative="greater")
    n1, n2 = len(a), len(b)
    rank_biserial = (2 * u_one.statistic) / (n1 * n2) - 1
    pooled_sd = np.sqrt(((n1-1)*a.std(ddof=1)**2 + (n2-1)*b.std(ddof=1)**2) / (n1+n2-2))
    cohens_d = (a.mean() - b.mean()) / pooled_sd
    print(f"\n{name_a} (n={n1}) vs {name_b} (n={n2}):")
    print(f"  Welch's t:      two-sided p={t_two.pvalue:.5f}   one-sided (greater) p={t_one.pvalue:.5f}")
    print(f"  Mann-Whitney:   two-sided p={u_two.pvalue:.5f}   one-sided (greater) p={u_one.pvalue:.5f}")
    print(f"  Direct Cohen's d: {cohens_d:+.3f}   Rank-biserial: {rank_biserial:+.3f}")
    clears_bonf = u_one.pvalue < BONFERRONI_ALPHA
    print(f"  Clears Bonferroni threshold ({BONFERRONI_ALPHA:.4f})? {'YES' if clears_bonf else 'no'}")
    return t_two.pvalue, t_one.pvalue, u_two.pvalue, u_one.pvalue, cohens_d, rank_biserial

print("=== Spatial vs Nonspatial-verb ===")
res_verb = compare("Spatial", spatial_sims, "Nonspatial-verb", verb_sims)

print("\n=== Spatial vs Nonspatial-modifier (load-bearing) ===")
res_modifier = compare("Spatial", spatial_sims, "Nonspatial-modifier", modifier_sims)

print("\n=== Nonspatial-verb vs Nonspatial-modifier (confound check) ===")
res_verb_vs_mod = compare("Nonspatial-verb", verb_sims, "Nonspatial-modifier", modifier_sims)

Spatial              (n=40): mean=0.9707  std=0.0149  min=0.9330  max=0.9932
Nonspatial-verb      (n=40): mean=0.9206  std=0.0394  min=0.8168  max=0.9899
Nonspatial-modifier  (n=40): mean=0.9442  std=0.0346  min=0.8131  max=0.9890

Kruskal-Wallis (all 3 groups, n=40 each): H=43.529, p=0.000000
=== Spatial vs Nonspatial-verb ===

Spatial (n=40) vs Nonspatial-verb (n=40):
  Welch's t:      two-sided p=0.00000   one-sided (greater) p=0.00000
  Mann-Whitney:   two-sided p=0.00000   one-sided (greater) p=0.00000
  Direct Cohen's d: +1.686   Rank-biserial: +0.779
  Clears Bonferroni threshold (0.0167)? YES

=== Spatial vs Nonspatial-modifier (load-bearing) ===

Spatial (n=40) vs Nonspatial-modifier (n=40):
  Welch's t:      two-sided p=0.00004   one-sided (greater) p=0.00002
  Mann-Whitney:   two-sided p=0.00001   one-sided (greater) p=0.00000
  Direct Cohen's d: +0.995   Rank-biserial: +0.591
  Clears Bonferroni threshold (0.0167)? YES

=== Nonspatial-verb vs Nonspatial-modifier (confound c

In [7]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 5))
lo = min(spatial_sims.min(), verb_sims.min(), modifier_sims.min()) - 0.02
hi = max(spatial_sims.max(), verb_sims.max(), modifier_sims.max()) + 0.02
bins = np.linspace(lo, hi, 26)
ax.hist(verb_sims, bins=bins, alpha=0.5, label=f"non-spatial verb (n={len(verb_sims)})", color="tab:blue")
ax.hist(modifier_sims, bins=bins, alpha=0.5, label=f"non-spatial modifier (n={len(modifier_sims)})", color="tab:green")
ax.hist(spatial_sims, bins=bins, alpha=0.5, label=f"spatial (n={len(spatial_sims)})", color="tab:red")
for arr, color in [(verb_sims, "tab:blue"), (modifier_sims, "tab:green"), (spatial_sims, "tab:red")]:
    ax.axvline(arr.mean(), color=color, linestyle="--", linewidth=1)
ax.set_xlabel("cosine similarity within minimal pair")
ax.set_ylabel("count")
ax.set_title(f"CLIP text-encoder similarity, n={len(spatial_sims)}/group")
ax.legend()
plt.tight_layout()
plt.savefig("01_clip_spatial_blindness_histogram.png", dpi=110)
plt.show()
print("saved 01_clip_spatial_blindness_histogram.png")

saved 01_clip_spatial_blindness_histogram.png


/var/folders/sb/x2_py571213939_gjl4zbhxc0000gn/T/ipykernel_28172/4049772929.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Corpus footprint (unchanged from round 1, term list already expanded per SUP-20260907-82)

In [8]:
import glob
import re

TEXTS_DIR = os.path.join(MDM_ROOT, "dataset", "HumanML3D", "texts")
SPATIAL_TERMS = ["left", "right", "forward", "forwards", "backward", "backwards",
                  "clockwise", "counterclockwise", "anticlockwise", "upleft",
                  "behind", "in front of"]

txt_files = sorted(glob.glob(os.path.join(TEXTS_DIR, "*.txt")))
print(f"{len(txt_files)} caption files found under {TEXTS_DIR}")

total_captions = 0
spatial_captions = 0
term_counts = {t: 0 for t in SPATIAL_TERMS}

for path in txt_files:
    with open(path, encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.strip()
            if not line or "#" not in line:
                continue
            caption = line.split("#")[0].strip().lower()
            if not caption:
                continue
            total_captions += 1
            hit = False
            for term in SPATIAL_TERMS:
                if re.search(r"\b" + re.escape(term) + r"\b", caption):
                    term_counts[term] += 1
                    hit = True
            if hit:
                spatial_captions += 1

pct = 100 * spatial_captions / total_captions if total_captions else float("nan")
print(f"\nTotal captions scanned: {total_captions}")
print(f"Captions containing at least one spatial term: {spatial_captions} ({pct:.1f}%)")
print("\nPer-term counts (captions can contain more than one term):")
for term, count in sorted(term_counts.items(), key=lambda kv: -kv[1]):
    print(f"  {term!r:20s} {count:6d}  ({100*count/total_captions:.2f}%)")

8198 caption files found under ../third_party/motion-diffusion-model/dataset/HumanML3D/texts



Total captions scanned: 24503
Captions containing at least one spatial term: 14112 (57.6%)

Per-term counts (captions can contain more than one term):
  'right'                5942  (24.25%)
  'forward'              5544  (22.63%)
  'left'                 5327  (21.74%)
  'backwards'             875  (3.57%)
  'in front of'           704  (2.87%)
  'clockwise'             633  (2.58%)
  'forwards'              346  (1.41%)
  'counterclockwise'      247  (1.01%)
  'backward'              243  (0.99%)
  'behind'                148  (0.60%)
  'anticlockwise'          17  (0.07%)
  'upleft'                  9  (0.04%)


## 6. Verdict — stated at the corrected, adequately-powered n

Reported honestly: if the effect vanished at n=40, that would be reported as vanishing, not
salvaged. The load-bearing test is spatial-vs-modifier at Bonferroni-corrected significance.

In [9]:
t_two_v, t_one_v, u_two_v, u_one_v, d_v, rb_v = res_verb
t_two_m, t_one_m, u_two_m, u_one_m, d_m, rb_m = res_modifier
t_two_vm, t_one_vm, u_two_vm, u_one_vm, d_vm, rb_vm = res_verb_vs_mod

verdict_lines = []

survives_bonferroni = u_one_m < BONFERRONI_ALPHA
survives_uncorrected = u_one_m < 0.05

if survives_bonferroni:
    verdict_lines.append(
        f"CONFIRMED at Bonferroni-corrected significance (n=40/group, adequately powered): "
        f"spatial pairs (mean {spatial_sims.mean():.4f}) are measurably LESS separated than "
        f"non-spatial modifier pairs in the same syntactic slot (mean {modifier_sims.mean():.4f}). "
        f"Mann-Whitney one-sided p={u_one_m:.5f} (two-sided p={u_two_m:.5f}), clears the "
        f"Bonferroni threshold ({BONFERRONI_ALPHA:.4f}) for the 3 comparisons run. "
        f"Direct Cohen's d={d_m:+.3f}."
    )
elif survives_uncorrected:
    verdict_lines.append(
        f"PARTIALLY CONFIRMED: spatial vs modifier clears uncorrected alpha=0.05 one-sided "
        f"(p={u_one_m:.5f}, two-sided p={u_two_m:.5f}) but NOT the Bonferroni-corrected threshold "
        f"({BONFERRONI_ALPHA:.4f}) for the 3 comparisons actually run. Reported as a real but "
        f"fragile signal at n=40, not a confident confirmation -- this is the honest reading, "
        f"not the headline reading."
    )
else:
    verdict_lines.append(
        f"NOT CONFIRMED at n=40: spatial vs modifier p={u_one_m:.5f} one-sided "
        f"(two-sided p={u_two_m:.5f}). The round-1 finding at n=16 does not survive the adequately "
        f"-powered re-run. This would be reported as the effect vanishing, not salvaged."
    )

verdict_lines.append(
    f"\nConfound check (verb vs modifier), two-sided p={u_two_vm:.5f}: " +
    ("still no difference between the two non-spatial groups -- the spatial effect above is "
     "attributable to spatial-ness alone, with no residual word-class confound to account for."
     if u_two_vm >= 0.05 else
     f"a real difference now shows between the two non-spatial groups (verb mean "
     f"{verb_sims.mean():.4f} vs modifier mean {modifier_sims.mean():.4f}, d={d_vm:+.3f}) -- "
     "at n=40, modifiers in general separate somewhat worse than verbs, so the clean "
     "'no confound at all' story from the n=16 pilot does not fully survive. This does NOT "
     "invalidate the primary spatial-vs-modifier result above: that comparison is already "
     "matched on word class (modifier vs modifier, same syntactic slot), so it does not depend "
     "on verbs and modifiers being equivalent -- it only needed the SAME non-spatial word class "
     "as the spatial group, which it has. The verb-vs-modifier gap is a separate, secondary "
     "finding (CLIP may separate modifiers worse than verbs generally, independent of "
     "spatial-ness) worth its own follow-up, not a defect in the primary test.")
)

verdict_lines.append(
    f"\nFor reference, spatial vs verb (the original, more separated comparison): one-sided "
    f"p={u_one_v:.6f}, two-sided p={u_two_v:.6f}, Cohen's d={d_v:+.3f}."
)

verdict_lines.append(
    f"\nCorpus footprint: {pct:.1f}% of {total_captions} HumanML3D captions contain at least one "
    f"of {SPATIAL_TERMS}. " +
    ("This is a narrow slice of the benchmark, not a dominant failure mode." if pct < 10 else
     "This is not a narrow footnote -- a majority of the benchmark's own captions use language "
     "this encoder may not distinguish.")
)

print("\n".join(verdict_lines))

CONFIRMED at Bonferroni-corrected significance (n=40/group, adequately powered): spatial pairs (mean 0.9707) are measurably LESS separated than non-spatial modifier pairs in the same syntactic slot (mean 0.9442). Mann-Whitney one-sided p=0.00000 (two-sided p=0.00001), clears the Bonferroni threshold (0.0167) for the 3 comparisons run. Direct Cohen's d=+0.995.

Confound check (verb vs modifier), two-sided p=0.00244: a real difference now shows between the two non-spatial groups (verb mean 0.9206 vs modifier mean 0.9442, d=-0.639) -- at n=40, modifiers in general separate somewhat worse than verbs, so the clean 'no confound at all' story from the n=16 pilot does not fully survive. This does NOT invalidate the primary spatial-vs-modifier result above: that comparison is already matched on word class (modifier vs modifier, same syntactic slot), so it does not depend on verbs and modifiers being equivalent -- it only needed the SAME non-spatial word class as the spatial group, which it has.